# Building Age Classification from Satellite Imagery

# Notebook 4 — Modelling & Evaluation

---

**Approach: Metric Learning**

In this notebook we will use a **metric learning** approach to the classification problem. This involves learning a feature representation where
buildings of the same age class cluster together, regardless of which city they come from. 

In practice this means:
1. **Train on Madrid** (the labelled city) using cross-validation to get reliable performance
   estimates with error bars.
2. **Zero-shot transfer** — apply the Madrid model directly to Amsterdam, with no Amsterdam
   labels, to measure the baseline domain gap.
3. **Few-shot prototype transfer** — given a small number of labelled Amsterdam examples,
   compute per-class prototype vectors (class means in feature space) and classify by
   nearest prototype. This is the inference step of prototype networks, a standard
   metric learning technique.

The few-shot experiment shows how much label budget is needed to close the Madrid–Amsterdam gap.

For more information on the metric learning approach, please see the supportind document "METRIC_LEARNING_APPROACH.md"

**NOTE**:  Metric learning is _one possible approach_.  Feel free to try another approach!

## 4.1 Setup & Imports

In [1]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from pathlib import Path

import tensorflow as tf
from tensorflow.keras import layers, models, losses, optimizers, callbacks
from sklearn.model_selection import StratifiedKFold, train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)

CLASS_COLORS = {1: '#4393c3', 2: '#2ca25f', 3: '#fd8d3c', 4: '#de2d26'}
CLASS_LABELS_MADRID    = {1: 'Pre-1960',  2: '1960-1984', 3: '1984-2004', 4: '2004-2024'}
CLASS_LABELS_AMSTERDAM = {1: 'Pre-1945',  2: '1945-1984', 3: '1984-2004', 4: '2004-2024'}

plt.rcParams['figure.dpi'] = 100
print("Setup complete.")


I0000 00:00:1792973257.367022  634639 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1792973257.742072  634639 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792973259.665424  634639 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Setup complete.


## 4.2 Load Preprocessed Data

In [2]:
with open('data/preprocessed/preprocessed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_madrid    = data['X_madrid']
y_madrid    = data['y_madrid']
X_amsterdam = data['X_amsterdam']
y_amsterdam = data['y_amsterdam']
feature_names = data['feature_names']
pixel_ids_madrid    = data['pixel_ids_madrid']     # (N, 2)  px_key, py_key
pixel_ids_amsterdam = data['pixel_ids_amsterdam']  # (N, 2)  px_key, py_key

print(f'Madrid:    {X_madrid.shape}  ({len(np.unique(y_madrid))} classes)')
print(f'Amsterdam: {X_amsterdam.shape}  ({len(np.unique(y_amsterdam))} classes)')
print(f'Features:  {len(feature_names)}')
print()
for name, y, labels in [('Madrid', y_madrid, CLASS_LABELS_MADRID),
                         ('Amsterdam', y_amsterdam, CLASS_LABELS_AMSTERDAM)]:
    print(f'{name} class distribution:')
    for cls in sorted(np.unique(y)):
        n = (y == cls).sum()
        print(f'  Class {cls} ({labels[cls]}): {n:,}  ({n/len(y)*100:.1f}%)')
    print()
    
smote = SMOTE(random_state=42)
X_madrid, y_madrid = smote.fit_resample(X_madrid, y_madrid)
y_madrid = y_madrid - 1
y_amsterdam = y_amsterdam - 1

Madrid:    (76263, 60)  (4 classes)
Amsterdam: (25992, 60)  (4 classes)
Features:  60

Madrid class distribution:
  Class 1 (Pre-1960): 13,794  (18.1%)
  Class 2 (1960-1984): 27,152  (35.6%)
  Class 3 (1984-2004): 18,672  (24.5%)
  Class 4 (2004-2024): 16,645  (21.8%)

Amsterdam class distribution:
  Class 1 (Pre-1945): 7,568  (29.1%)
  Class 2 (1945-1984): 8,802  (33.9%)
  Class 3 (1984-2004): 6,745  (26.0%)
  Class 4 (2004-2024): 2,877  (11.1%)



In [3]:
NUM_CLASSES = len(np.unique(y_madrid))

In [4]:
# create a function that returns a model instance 
def create_model():
    model = models.Sequential([
        layers.Input(shape=(X_madrid.shape[1],)),
        
        layers.Dense(128, activation='relu', name='madrid_dense_1'),
        layers.BatchNormalization(),
        layers.Dropout(rate=0.3),

        layers.Dense(64, activation='relu', name='madrid_dense_2'),
        layers.BatchNormalization(),
        layers.Dropout(rate=0.2),
        
        layers.Dense(32, activation='relu', name='madrid_dense_3'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),

        layers.Dense(NUM_CLASSES, activation='softmax')
    ])

    model.compile(
        optimizer = optimizers.Adam(learning_rate=0.005),
        # using sparse cross entropy since y contains interger labels 
        loss = losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy']
    )

    return model

# cv = StratifiedKFold(
#     n_splits=5,
#     shuffle=True,
#     random_state=42
# )

# fold_accuracies = []
# fold_fi_scores = []

# # continueross validation loop 
# for fold, (train_ids, val_ids) in enumerate(cv.split(X_madrid, y_madrid)):
#     print(f"\n--- Fold {fold + 1} ---")

#     # Split arrays by current indices 
#     X_train, X_val = X_madrid[train_ids], X_madrid[val_ids]
#     y_train, y_val = y_madrid[train_ids], y_madrid[val_ids]

#     # Instantiate a clean architecture and train 
#     model = create_model()

#     # Define EarlyStopping 
#     early_stopping = callbacks.EarlyStopping(
#         monitor='val_loss',
#         patience=3,
#         restore_best_weights=True
#     )
#     history = model.fit(
#         X_train, y_train, 
#         validation_data =(X_val, y_val),
#         epochs=100,
#         batch_size=64,
#         callbacks=[early_stopping],
#         verbose=0
#     )

#     # Standard evaluation (loss and accuracy)
#     scores = model.evaluate(X_val, y_val, verbose=0)
#     val_acc = scores[1]

#     # Macro_f1 evaluation step 
#     # get probability arrays from the network 
#     y_pred_probs = model.predict(X_val, verbose=0)
#     # convert probabilities to distinct classes 
#     y_pred = np.argmax(y_pred_probs, axis=1)

#     # Calculate macro_f1 comparing the correct labels to predictions 
#     val_f1 = f1_score(y_val, y_pred, average="macro")

#     print(f"Fold {fold + 1} - Accuracy: {val_acc:.4f} | Macro_F1: {val_f1:.4f}")

#     fold_accuracies.append(val_acc)
#     fold_fi_scores.append(val_f1)

#     # Clean up graph state to prevent slowdowns 
#     tf.keras.backend.clear_session()

# # Aggregated performance across all folds 
# print("\n--- FINAL RESULTS ---")
# print(f"Average CV Accuracy: {np.mean(fold_accuracies):.4f} +/- {np.std(fold_accuracies)}")
# print(f"Average Macro F1: {np.mean(fold_fi_scores):.4f} +/- {np.std(fold_fi_scores):.4f}")

### Fit the model on the full dataset

In [5]:
# fit the model with the entire dataset 
def build_model():
    inputs = layers.Input(shape=(X_madrid.shape[1],))
    
    x = layers.Dense(128, name='madrid_dense_1')(inputs)
    x = layers.LeakyReLU(negative_slope=0.1)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(rate=0.3)(x)

    feature_output = layers.Dense(64, name='madrid_dense_2')(x)
    feature_output = layers.LeakyReLU(negative_slope=0.1)(feature_output)
    x = layers.BatchNormalization()(feature_output)
    x = layers.Dropout(rate=0.2)(x)
    
    feature_output_2 = layers.Dense(32, name='madrid_dense_3')(x)
    feature_output_2 = layers.LeakyReLU(negative_slope=0.1)(feature_output_2)
    x = layers.BatchNormalization()(feature_output_2)
    x = layers.Dropout(0.2)(x)

    outputs = layers.Dense(NUM_CLASSES, activation='softmax', name="output")(x)

    model = models.Model(inputs=inputs, outputs=outputs)

    model.compile(
        optimizer = optimizers.Adam(learning_rate=0.005),
        # using sparse cross entropy since y contains interger labels 
        loss = losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy']
    )

    return model

madrid_model = build_model()

early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)
madrid_model.fit(
    X_madrid, y_madrid, 
    validation_split=0.2,
    epochs=100,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=0
)

# Extract the feature layers (everything except the final output)
feature_extractor = models.Model(
    inputs=madrid_model.input,
    outputs=madrid_model.get_layer("madrid_dense_3").output
)

results = {}

# Zero-shot Evaluation 
print("\n=== Evaluating Zero-Shot Transfer ===")
# Directly predict Amsterdam data using the out-of-the-box Madrid model 
y_pred_probs_zero = madrid_model.predict(X_amsterdam, verbose=0)
y_pred_zero = np.argmax(y_pred_probs_zero, axis=1)
zero_shot_f1 = f1_score(y_amsterdam, y_pred_zero, average="macro")

results[0] = zero_shot_f1
print(f"Zero-Shot Amsterdam Macro F1: {zero_shot_f1:.4f}")
madrid_model.summary()

E0000 00:00:1792973263.374675  634639 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
W0000 00:00:1792973263.515403  634639 cpu_allocator_impl.cc:82] Allocation of 20852640 exceeds 10% of free system memory.



=== Evaluating Zero-Shot Transfer ===
Zero-Shot Amsterdam Macro F1: 0.4892


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 60)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ madrid_dense_1 (Dense)          │ (None, 128)            │         7,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ madrid_dense_2 (Dense)          │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ madrid_dense_3 (Dense)          │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_2 (LeakyReLU)       │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 56,622 (221.18 KB)

 Trainable params: 18,724 (73.14 KB)

 Non-trainable params: 448 (1.75 KB)

 Optimizer params: 37,450 (146.29 KB)

In [6]:
# FEW-SHOT BENCHMARK LOOP 
few_shot_sizes = [5, 10, 25, 50, 100, 200]

def build_transfer_head(base_extractor):
    # Clone weights to isolate runs completely 
    base_copy = models.model_from_json(base_extractor.to_json())
    base_copy.set_weights(base_extractor.get_weights())
    base_copy.trainable = False

    model = models.Sequential([
        base_copy,
        layers.Dense(16, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ])

    model.compile(
        optimizer = optimizers.Adam(learning_rate=0.002),
        loss = losses.SparseCategoricalCrossentropy()
    )

    return model

print("\n=== Evaluating Few-Shot Transfer ===")
for n in few_shot_sizes:
    if n >= len(X_amsterdam):
        continue

    X_shot, X_test_val, y_shot, y_test_val = train_test_split(
        X_amsterdam, y_amsterdam,
        train_size=n,
        stratify=y_amsterdam,
        random_state=42
    )

    # Initialize a fresh transfer model instance 
    transfer_model = build_transfer_head(feature_extractor)

    # train exclusively on the few-shot subset 
    early_stopping = callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )

    transfer_model.fit(
        X_shot, y_shot,
        validation_data=(X_test_val, y_test_val),
        epochs=50,
        batch_size=min(8, n),
        callbacks=[early_stopping],
        verbose=0
    )

    # Evaluate performance on large remaining unseen portion of Amsterdam data 
    y_pred_probs = transfer_model.predict(X_test_val, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    macro_f1 = f1_score(y_test_val, y_pred, average='macro')

    results[n] = macro_f1
    print(f"Few-Shot (N={n:3d}) Amsterdam Macro F1: {macro_f1:.4f}")

    tf.keras.backend.clear_session()

# FINAL COMPARATIVE SUMMARY 
print("\n=== FINAL TRANSFER PERFORMANCE SUMMARY ===")
print(f"{'Samples (N)': <15}{'Macro F1 Score':<15}")
print("-"*30)

for shot, score in results.items():
    label = "Zero-Shot" if shot == 0 else f"{shot}-Shot"
    print(f"{label:<15}{score:.4f}")


=== Evaluating Few-Shot Transfer ===
Few-Shot (N=  5) Amsterdam Macro F1: 0.3323
Few-Shot (N= 10) Amsterdam Macro F1: 0.4465
Few-Shot (N= 25) Amsterdam Macro F1: 0.5739
Few-Shot (N= 50) Amsterdam Macro F1: 0.5962
Few-Shot (N=100) Amsterdam Macro F1: 0.5978
Few-Shot (N=200) Amsterdam Macro F1: 0.6320

=== FINAL TRANSFER PERFORMANCE SUMMARY ===
Samples (N)    Macro F1 Score 
------------------------------
Zero-Shot      0.4892
5-Shot         0.3323
10-Shot        0.4465
25-Shot        0.5739
50-Shot        0.5962
100-Shot       0.5978
200-Shot       0.6320
